In [25]:
# pip install scikit-clean pandas numpy sklearn

# Importo librerías

In [26]:
import numpy as np
import pandas as pd
rs=33

# Funciones generación de datasets ruidosos

In [27]:
from numpy.random import randint, seed
def urlf(y, noise_level=0.1, random_state=42):
    """
    UniformRandomizedLabelFlip (corregida)

    - No modifica y original (trabaja sobre copia).
    - Cambia un % de etiquetas a otra etiqueta (uniforme).
    - Funciona con etiquetas arbitrarias (strings, ints, etc.)
    """
    y = np.asarray(y)
    y_out = y.copy()

    seed(random_state)

    classes, y_idx = np.unique(y_out, return_inverse=True)
    n = len(y_out)
    k = int(noise_level * n)
    if k <= 0:
        return y_out

    idx_to_change = randint(low=0, high=n, size=k)
    # nuevos índices de clase (0..K-1)
    new_idx = randint(low=0, high=len(classes), size=len(idx_to_change))

    # opcional: asegurar "flip" real (que no caiga la misma clase)
    # (si no te importa que a veces se quede igual, comenta estas 4 líneas)
    same = new_idx == y_idx[idx_to_change]
    if np.any(same) and len(classes) > 1:
        new_idx[same] = (new_idx[same] + 1) % len(classes)

    y_out[idx_to_change] = classes[new_idx]
    return y_out

# Funciones testeo modelos

In [56]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler

def urlf_test_in_dfs(
    dfs, 
    dfs_names, 
    noise_levels, 
    rs=33, 
    filter = None, 
    model = RandomForestClassifier(random_state=33, n_jobs=-1),
    sc = StandardScaler(),
    k_cv=5
    ):

    # Initialize dict to store results
    res = {
        "df_name":[],
        "noise_pct":[],
        "Acc":[],
        "BalAcc":[],
        "f1_macro":[],
        "Prec_macro":[],
        "Rec_macro":[]
    }

    # Iter through dataframes
    for (df_name, df) in zip(dfs_names,dfs):

        # Extract attributes and target from df
        X = df.iloc[:,:-1].values
        y = df.iloc[:,-1].values

        # First compute baseline (no filter nor noise) results with df data
        cv = cross_validate(
            estimator=Pipeline(
                [
                    ("sc", sc),
                    ("model", model),
                ]
            ),
            X=X,
            y=y,
            scoring={
                "Acc":"accuracy",
                "BalAcc":"balanced_accuracy",
                "f1_macro":"f1_macro",
                "Prec_macro":"precision_macro",
                "Rec_macro":"recall_macro"
            },
            n_jobs=-1,
            cv=k_cv)

        # Store results
        res["df_name"].append(df_name)
        res["noise_pct"].append(-1)    # -1 means baseline with no filtering technique applied
        res["Acc"].append(cv["test_Acc"].mean())
        res["BalAcc"].append(cv["test_BalAcc"].mean())
        res["f1_macro"].append(cv["test_f1_macro"].mean())
        res["Prec_macro"].append(cv["test_Prec_macro"].mean())
        res["Rec_macro"].append(cv["test_Rec_macro"].mean())

        # Iter through noise_levels
        for nl in noise_levels:
            print(f"Processing {df_name} with noise level={nl}.")
            # Apply random uniform noise with nl as noise level
            y_noisy = urlf(y, noise_level=nl, random_state=rs)

            # Compute results without filter
            cv = cross_validate(
                estimator=Pipeline(
                    [
                        ("sc", sc),
                        ("model", model),
                    ]
                ),
                X=X,
                y=y_noisy,
                scoring={
                    "Acc":"accuracy",
                    "BalAcc":"balanced_accuracy",
                    "f1_macro":"f1_macro",
                    "Prec_macro":"precision_macro",
                    "Rec_macro":"recall_macro"
                },
                n_jobs=-1,
                cv=k_cv)

            # Store results
            res["df_name"].append(df_name+"_nf")   # nf means "not filtered"
            res["noise_pct"].append(nl)    # -1 means baseline with no filtering technique applied
            res["Acc"].append(cv["test_Acc"].mean())
            res["BalAcc"].append(cv["test_BalAcc"].mean())
            res["f1_macro"].append(cv["test_f1_macro"].mean())
            res["Prec_macro"].append(cv["test_Prec_macro"].mean())
            res["Rec_macro"].append(cv["test_Rec_macro"].mean())

            # Compute results with filter applied
            cv = cross_validate(
                estimator=Pipeline(
                    [
                        ("filter", filter),
                        ("sc", sc),
                        ("model", model),
                    ]
                ),
                X=X,
                y=y_noisy,
                scoring={
                    "Acc":"accuracy",
                    "BalAcc":"balanced_accuracy",
                    "f1_macro":"f1_macro",
                    "Prec_macro":"precision_macro",
                    "Rec_macro":"recall_macro"
                },
                n_jobs=-1,
                cv=k_cv)

            # Store results
            res["df_name"].append(df_name+"_f")   # f means "filtered"
            res["noise_pct"].append(nl)    # -1 means baseline with no filtering technique applied
            res["Acc"].append(cv["test_Acc"].mean())
            res["BalAcc"].append(cv["test_BalAcc"].mean())
            res["f1_macro"].append(cv["test_f1_macro"].mean())
            res["Prec_macro"].append(cv["test_Prec_macro"].mean())
            res["Rec_macro"].append(cv["test_Rec_macro"].mean())
        print("\n")

    return res

# Preparo datos

Para testar las técnicas a definir a continuación crearemos a partir de un dataset limpio otro con ruido:

In [29]:
from sklearn.datasets import load_breast_cancer

clean_ds = pd.concat(load_breast_cancer(return_X_y=True, as_frame=True), axis=1)
clean_ds.sample(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
474,10.88,15.62,70.41,358.9,0.1007,0.1069,0.05115,0.01571,0.1861,0.06837,...,19.35,80.78,433.1,0.1332,0.3898,0.3365,0.07966,0.2581,0.10800,1
408,17.99,20.66,117.80,991.7,0.1036,0.1304,0.12010,0.08824,0.1992,0.06069,...,25.41,138.10,1349.0,0.1482,0.3735,0.3301,0.19740,0.3060,0.08503,0
89,14.64,15.24,95.77,651.9,0.1132,0.1339,0.09966,0.07064,0.2116,0.06346,...,18.24,109.40,803.6,0.1277,0.3089,0.2604,0.13970,0.3151,0.08473,1


In [30]:
noisy_ds = clean_ds.copy()
noisy_ds.target = urlf(noisy_ds.target.to_numpy())

In [31]:
y = noisy_ds.target.to_numpy()
y = urlf(y)
(y==clean_ds.target.to_numpy()).mean()

np.float64(1.0)

Si quisiéramos comprobar la correcta randomización:

In [32]:
(noisy_ds.target==clean_ds.target).mean()

np.float64(0.9050966608084359)

# Diseño filtros

In [46]:
import numpy as np
from dataclasses import dataclass
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.validation import check_X_y

@dataclass
class EnsembleFilterResult:
    keep_mask: np.ndarray
    noisy_fraction: float
    noisy_votes: np.ndarray  # nº de modelos que discrepan con y
    n_models: int

class EnsembleFiltering(BaseEstimator, TransformerMixin):
    """
    Ensemble Filtering (label-noise filtering) sklearn-compatible.

    Idea: train multiple base classifiers with CV, obtain out-of-fold predictions,
    and mark an instance as 'noisy' if many classifiers disagree with its label.
    Then remove it (default) or optionally relabel it with the ensemble vote.

    Parameters
    ----------
    estimators : list
        List of sklearn classifiers (must implement fit/predict).
    cv : int
        Number of folds for out-of-fold predictions.
    mode : {"majority", "consensus"}
        - "majority": remove if (#wrong_votes / n_models) >= threshold (default threshold=0.5)
        - "consensus": remove if all models disagree with y (threshold ignored)
    threshold : float
        Only used for mode="majority". Typical: 0.5 or 0.6.
    action : {"remove", "relabel"}
        - "remove": drop flagged instances
        - "relabel": keep all, but replace y of flagged instances with ensemble majority vote
    random_state : int
        Seed for CV splits.
    """
    def __init__(
        self,
        estimators,
        cv=10,
        mode="majority",
        threshold=0.5,
        action="remove",
        random_state=33,
        return_noisy_samples=False, 
    ):
        self.estimators = estimators
        self.cv = cv
        self.mode = mode
        self.threshold = threshold
        self.action = action
        self.random_state = random_state
        self.return_noisy_samples = return_noisy_samples

    def fit(self, X, y):
        # Check correct form of X,y
        X, y = check_X_y(X, y, accept_sparse=True)
        # Get number of differente classes and turn y to ordinal (with classes 0,1,2,...)
        self.classes_, y_idx = np.unique(y, return_inverse=True)
        n_classes = len(self.classes_)
        # Get number of samples
        n = X.shape[0]
        # Get number of estimators
        m = len(self.estimators)
        if m < 2:
            raise ValueError("Provide at least 2 estimators for ensemble filtering.")
        # Initialize StratifiedKFold strategy
        skf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=self.random_state)

        # Initialize OutOfFold prediction matrix (m rows, n cols) -> (n_models, n_samples)
        oof_preds = np.empty((m, n), dtype=int)
        # Fill OutOfFold prediction matrix
        for est_idx, est in enumerate(self.estimators):
            for train_indices, test_indices in skf.split(X, y_idx):
                # Clone the estimator from zero (to avoid refitting)
                model = clone(est)
                # Fit the model with all folds but one out
                model.fit(X[train_indices], y_idx[train_indices])
                # Predict labels on the fold left above
                oof_preds[est_idx, test_indices] = model.predict(X[test_indices])
                
        # Compare each set of estimator predicitons (rows of oof_preds) to the real ones (y_idx)
        # and compute the pct of missclassification for each sample over all the estimators
        wrong_votes = (oof_preds != y_idx[None, :]).sum(axis=0)  # (n,)
        wrong_frac = wrong_votes / m

        # Compute action mask based on self.action
        if self.mode == "consensus":
            noisy_mask = (wrong_votes == m)
        elif self.mode == "majority":
            noisy_mask = (wrong_frac >= self.threshold)
        else:
            raise ValueError("mode must be 'majority' or 'consensus'")

        # Compute mask of samples to keep
        keep_mask = ~noisy_mask

        # TODO: Añadir apartado relabel, quizá compatible con otras librerías o similar
        # # If relabel: replace noisy labels by ensemble majority vote (mode)
        # self.y_relabel_ = None
        # if self.action == "relabel":
        #     # majority vote prediction per sample
        #     # bincount per column (fast enough for moderate sizes)
        #     y_new = y_idx.copy()
        #     for i in np.where(noisy_mask)[0]:
        #         counts = np.bincount(oof_preds[:, i], minlength=n_classes)
        #         y_new[i] = counts.argmax()
        #     self.y_relabel_ = self.classes_[y_new]

        self.result_ = EnsembleFilterResult(
            keep_mask=keep_mask,
            noisy_fraction=float(noisy_mask.mean()),
            noisy_votes=wrong_votes,
            n_models=m,
        )
        self.X_ = X
        self.y_ = y
        return self

    def transform(self, X):
        # Turn X to array if it's not yet
        X = np.asarray(X) if not hasattr(X, "shape") else X

        # Remove noisy_samples following self.mode criterion if self.action=="remove"
        if self.action == "remove":
            return X[self.result_.keep_mask]

        return X # TODO: Añadir que pasa si relabel

    def fit_resample(self, X, y):
        # Fit the ensmble filter
        self.fit(X, y)
        # Once fitted return X,y following self.mode criterion
        if self.action == "remove":
            return self.X_[self.result_.keep_mask], self.y_[self.result_.keep_mask]
        else:  # relabel
            return self.X_, self.y_relabel_

    def get_filter_report(self):
        return {
            "n_samples": int(self.X_.shape[0]),
            "n_models": int(self.result_.n_models),
            "removed_or_flagged": int((~self.result_.keep_mask).sum()),
            "fraction_flagged": float(self.result_.noisy_fraction),
            "mode": self.mode,
            "threshold": self.threshold if self.mode == "majority" else None,
            "action": self.action,
        }


In [69]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

from numpy.random import randint, seed


# --- Adapter: sampler "puro" para imblearn Pipeline ---
class EnsembleFilteringSampler(BaseEstimator):
    def __init__(self, estimators, cv=10, mode="majority", threshold=0.5, action="remove", random_state=33):
        self.estimators = estimators
        self.cv = cv
        self.mode = mode
        self.threshold = threshold
        self.action = action
        self.random_state = random_state
        self._filt = None

    def _build(self):
        return EnsembleFiltering(
            estimators=self.estimators,
            cv=self.cv,
            mode=self.mode,
            threshold=self.threshold,
            action=self.action,
            random_state=self.random_state,
        )

    def fit(self, X, y):
        self._filt = self._build()
        self._filt.fit(X, y)
        return self

    def fit_resample(self, X, y):
        self._filt = self._build()
        return self._filt.fit_resample(X, y)

    def get_filter_report(self):
        if self._filt is None:
            return None
        return self._filt.get_filter_report()


# --- construir el filtro (wrapper) ---
ens_filter_sampler = EnsembleFilteringSampler(
    estimators=[
        GaussianNB(),
        RandomForestClassifier(random_state=33),
    ],
    cv=5,
    mode="majority",
    threshold=0.75,
    action="remove",
    random_state=33,
)

# --- ejecutar TU función exactamente ---
noise_levels = [0.0, 0.05, 0.10, 0.20, 0.30]

res = urlf_test_in_dfs(
    dfs=[clean_ds],
    dfs_names=["ds"],
    noise_levels=noise_levels,
    rs=33,
    filter=ens_filter_sampler,  # <-- aquí el wrapper
    model=KNeighborsClassifier(),
    sc=StandardScaler(),
    k_cv=5
)

df_res = pd.DataFrame(res)
df_res

Processing ds with noise level=0.0.
Processing ds with noise level=0.05.
Processing ds with noise level=0.1.
Processing ds with noise level=0.2.
Processing ds with noise level=0.3.




,df_name,noise_pct,Acc,BalAcc,f1_macro,Prec_macro,Rec_macro
0,ds,-1.00,0.964850,0.958548,0.962058,0.967163,0.958548
1,ds_nf,0.00,0.964850,0.958548,0.962058,0.967163,0.958548
2,ds_f,0.00,0.964866,0.955741,0.961804,0.970142,0.955741
3,ds_nf,0.05,0.906816,0.895681,0.900727,0.909177,0.895681
4,ds_f,0.05,0.919127,0.905053,0.913214,0.927951,0.905053
5,ds_nf,0.10,0.854091,0.840540,0.845741,0.858456,0.840540
6,ds_f,0.10,0.864617,0.846909,0.854960,0.877714,0.846909
7,ds_nf,0.20,0.752166,0.734154,0.736803,0.755032,0.734154
8,ds_f,0.20,0.783807,0.764427,0.769265,0.796452,0.764427
9,ds_nf,0.30,0.676681,0.665554,0.663414,0.684381,0.665554
